# Notebook 08: Multi-Outcome Integration and Data Cleaning

The feature expansion in notebook 07 produced a 24-column dataset with strong child 
and maternal predictors. Before feature engineering begins, two data quality issues 
carried forward from that notebook need to be resolved, and the dataset needs to carry 
all three nutrition outcomes this project models simultaneously.

The two issues to fix are a string-encoded nan in str_wealth_index affecting 292 
records linked to the Dzaleka refugee camp domain, and 57 size-at-birth records that 
failed to collapse into not_reported due to a non-standard apostrophe character in 
the Stata encoding.

The three outcomes are stunting, wasting, and anemia. All three share upstream 
determinants in the UNICEF framework but diverge at the immediate cause level. 
Stunting is chronic linear growth failure accumulating over months. Wasting is acute 
weight loss responding to recent food insecurity or illness. Anemia has a distinct 
pathway through malaria exposure, iron deficiency, and inflammation. Modeling them 
simultaneously and identifying shared versus unique drivers is the core methodological 
contribution of this project.

District is added as a structural variable for the sub-national SHAP decomposition 
objective. Dzaleka refugee camp records are flagged for exclusion from national models 
as required by the 2024 MDHS sampling design, which treated Dzaleka as a separate 
sampling domain.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT   = Path("/Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai")
DATA_INTERIM   = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df_kr = pd.read_parquet(DATA_INTERIM / "kr_clean.parquet")
df    = pd.read_parquet(DATA_PROCESSED / "model_dataset_v2.parquet")

print("KR file:", df_kr.shape)
print("Model v2:", df.shape)
print("\nColumns loaded:")
print(df.columns.tolist())

KR file: (5415, 1210)
Model v2: (5414, 24)

Columns loaded:
['id_cluster', 'id_household', 'id_child_index', 'haz_score', 'outcome_stunted', 'wt_sample_weight', 'und_maternal_age', 'und_maternal_edu_level', 'str_wealth_index', 'str_residence', 'und_household_size', 'imm_child_age_months', 'imm_child_sex', 'imm_birth_order', 'imm_birth_interval', 'imm_size_at_birth', 'imm_had_diarrhea', 'und_maternal_edu_years', 'und_maternal_weight_kg', 'und_maternal_height_cm', 'und_total_children', 'str_region', 'str_religion', 'imm_first_born']


## Fixing data quality issues from notebook 07

Two issues are resolved here before any new variables are added.

The wealth index column has 292 records stored as the string "nan" rather than a 
proper null. These correspond to the Dzaleka refugee camp records which were assigned 
no wealth index in the national classification. Converting to a real null lets 
downstream code handle them deliberately.

Size at birth has 57 records still showing as "don't know" rather than "not_reported". 
The replace in notebook 07 used a standard apostrophe but the Stata encoding uses a 
right single quotation mark. A contains-based match catches it regardless of the 
exact character used.

In [2]:
# fix wealth index string nan from Dzaleka records
df["str_wealth_index"] = df["str_wealth_index"].replace("nan", np.nan)

print("Wealth index after fix:")
print(df["str_wealth_index"].value_counts(dropna=False))

# fix size at birth apostrophe encoding issue using contains match
df["imm_size_at_birth"] = df["imm_size_at_birth"].apply(
    lambda x: "not_reported" if "know" in str(x).lower() else x
)

print("\nSize at birth after fix:")
print(df["imm_size_at_birth"].value_counts())

Wealth index after fix:
str_wealth_index
poorest    1150
richest    1093
richer     1041
poorer      945
middle      893
NaN         292
Name: count, dtype: int64

Size at birth after fix:
imm_size_at_birth
not_reported            2077
average                 1610
larger than average     1031
smaller than average     374
very large               228
very small                94
Name: count, dtype: int64


## Adding district and Dzaleka flag

District provides sub-national resolution beyond region and is required for the 
SHAP decomposition objective where we identify which risk factors dominate across 
Malawi's 28 districts and 4 cities.

The merge uses the same four-part child key established in notebook 07: cluster, 
household, child index, and HAZ score. This avoids the positional length mismatch 
caused by the three twin records dropped during the notebook 07 merge.

The Dzaleka refugee camp was a separate sampling domain in the 2024 MDHS and its 
records are explicitly excluded from national estimates in the official report. We 
flag rather than drop them, preserving the option to analyse the camp population 
as a distinct stratum. These 292 records are the same ones with missing wealth index, 
confirming the connection between the two issues.

In [3]:
# build district lookup using original kr column names
kr_district = df_kr[["v001", "v002", "bidx", "hw70", "sdistrict"]].copy()

kr_district = kr_district.rename(columns={
    "v001"     : "id_cluster",
    "v002"     : "id_household",
    "bidx"     : "id_child_index",
    "hw70"     : "haz_score",
    "sdistrict": "str_district"
})

df = df.merge(
    kr_district,
    on=["id_cluster", "id_household", "id_child_index", "haz_score"],
    how="left"
)

before = len(df)
df = df.drop_duplicates(
    subset=["id_cluster", "id_household", "id_child_index", "haz_score"]
)
print(f"Shape after district merge: {len(df)} (dropped {before - len(df)})")

print("\nDistrict distribution:")
print(df["str_district"].value_counts())

df["flag_dzaleka"] = (df["str_district"] == "dowa (camps)").astype(int)
print(f"\nDzaleka records flagged: {df['flag_dzaleka'].sum()}")
print(f"National sample records: {(df['flag_dzaleka'] == 0).sum()}")

Shape after district merge: 5414 (dropped 1)

District distribution:
str_district
dowa (camps)     292
mangochi         219
salima           215
machinga         205
mzimba           192
zomba            192
kasungu          183
karonga          180
chikwawa         178
phalombe         176
mulanje          176
lilongwe         174
nsanje           168
nkhotakota       166
ntcheu           162
blantyre         160
thyolo           157
dedza            156
rumphi           153
chiradzulu       151
chitipa          150
balaka           148
likoma           147
nkhata bay       145
mwanza           143
mchinji          142
ntchisi          141
neno             135
dowa             134
mzuzu city       134
lilongwe city    122
blantyre city    114
zomba city       104
Name: count, dtype: int64

Dzaleka records flagged: 292
National sample records: 5122


## Adding wasting and anemia outcomes

Three anthropometric z-scores and two outcome variables are pulled from kr_clean 
using the four-part key merge to avoid any positional mismatch.

DHS coding for z-scores: hw72 (WHZ) and hw71 (WAZ) are stored as integers in 
hundredths, so -22 means -0.22. Both are divided by 100. Hemoglobin hw56 is stored 
in tenths of g/dL, so 108 means 10.8 g/dL, and is divided by 10.

Wasting binary outcome uses the WHO threshold of WHZ below -2. DHS flags implausible 
WHZ values with the label "flagged cases" and these are treated as missing.

Anemia uses hw57, the categorical severity classification. We collapse mild, moderate, 
and severe into a single binary anemic category. The 1,353 children with no 
hemoglobin measurement are left as null since missing measurement is informative 
about healthcare access and should not be imputed.

WAZ is retained as a continuous variable alongside the binary wasting outcome for 
any regression-based extensions.

In [4]:
# build outcome lookup from kr_clean
kr_outcomes = df_kr[["v001", "v002", "bidx", "hw70",
                      "hw72", "hw71", "hw57", "hw56"]].copy()

# WHZ and WAZ stored as integers in hundredths, divide by 100
kr_outcomes["hw72"] = pd.to_numeric(
    kr_outcomes["hw72"].replace("flagged cases", np.nan),
    errors="coerce"
) / 100

kr_outcomes["hw71"] = pd.to_numeric(
    kr_outcomes["hw71"], errors="coerce"
) / 100

# hemoglobin stored in tenths of g/dL, divide by 10
kr_outcomes["hw56"] = pd.to_numeric(
    kr_outcomes["hw56"].replace("nan", np.nan),
    errors="coerce"
) / 10

# anemia: collapse severity levels into binary outcome
kr_outcomes["hw57"] = (kr_outcomes["hw57"]
                       .astype(str).str.strip().str.lower()
                       .map({"not anemic": 0, "mild": 1,
                             "moderate": 1, "severe": 1}))

kr_outcomes = kr_outcomes.rename(columns={
    "v001" : "id_cluster",
    "v002" : "id_household",
    "bidx" : "id_child_index",
    "hw70" : "haz_score",
    "hw72" : "whz_score",
    "hw71" : "waz_score",
    "hw57" : "outcome_anemia",
    "hw56" : "hemoglobin_gdl"
})

kr_outcomes["outcome_wasting"] = (kr_outcomes["whz_score"] < -2).astype("Int64")
kr_outcomes["outcome_anemia"]  = pd.array(
    kr_outcomes["outcome_anemia"], dtype="Int64"
)

# merge on four-part key
df = df.merge(
    kr_outcomes,
    on=["id_cluster", "id_household", "id_child_index", "haz_score"],
    how="left"
)

before = len(df)
df = df.drop_duplicates(
    subset=["id_cluster", "id_household", "id_child_index", "haz_score"]
)
print(f"Shape after outcome merge: {len(df)} (dropped {before - len(df)})")

national = df[df["flag_dzaleka"] == 0]

print("\n=== Wasting ===")
print(df["outcome_wasting"].value_counts(dropna=False))
print(f"Wasting prevalence (national): {national['outcome_wasting'].mean():.3f}")

print("\n=== Anemia ===")
print(df["outcome_anemia"].value_counts(dropna=False))
print(f"Anemia prevalence (national): {national['outcome_anemia'].mean():.3f}")

print("\n=== WHZ summary ===")
print(df["whz_score"].describe().round(3))

print("\n=== WAZ summary ===")
print(df["waz_score"].describe().round(3))

print("\n=== Hemoglobin ===")
print(df["hemoglobin_gdl"].describe().round(2))

Shape after outcome merge: 5414 (dropped 1)

=== Wasting ===
outcome_wasting
0    5297
1     117
Name: count, dtype: Int64
Wasting prevalence (national): 0.021

=== Anemia ===
outcome_anemia
0       2101
1       1960
<NA>    1353
Name: count, dtype: Int64
Anemia prevalence (national): 0.489

=== WHZ summary ===
count    5370.000
mean        0.364
std         1.149
min        -4.850
25%        -0.330
50%         0.360
75%         1.080
max         4.950
Name: whz_score, dtype: float64

=== WAZ summary ===
count    5410.000
mean       -0.628
std         1.080
min        -5.710
25%        -1.310
50%        -0.640
75%         0.050
max         4.830
Name: waz_score, dtype: float64

=== Hemoglobin ===
count    4061.00
mean       10.91
std         1.49
min         4.00
25%        10.00
50%        11.00
75%        11.90
max        16.00
Name: hemoglobin_gdl, dtype: float64


## Final validation and save

Five checks run before saving: row count within expected range, missingness only 
in expected columns, sampling weight present on every record, all three outcomes 
present, and prevalence figures within the range reported in the 2024 MDHS official 
report.

The small differences from official figures are expected. The official report uses 
survey weights for prevalence estimation while these figures are unweighted. Survey 
weights are applied at the modeling stage. The dataset saves as model_dataset_v3.parquet. 
All previous versions remain untouched.

In [5]:
# reorder columns logically before saving
col_order = [
    "id_cluster", "id_household", "id_child_index",
    "wt_sample_weight", "flag_dzaleka",
    "haz_score", "whz_score", "waz_score", "hemoglobin_gdl",
    "outcome_stunted", "outcome_wasting", "outcome_anemia",
    "imm_child_age_months", "imm_child_sex", "imm_birth_order",
    "imm_birth_interval", "imm_first_born", "imm_size_at_birth",
    "imm_had_diarrhea",
    "und_maternal_age", "und_maternal_edu_level", "und_maternal_edu_years",
    "und_maternal_weight_kg", "und_maternal_height_cm",
    "und_total_children", "und_household_size",
    "str_wealth_index", "str_residence",
    "str_region", "str_district", "str_religion"
]
df = df[col_order]

assert len(df) in [5412, 5413, 5414, 5415], \
    f"Unexpected row count: {len(df)}"

print("=== Row count ===")
print(len(df))

print("\n=== Missingness ===")
miss = df.isnull().mean().round(3)
print(miss[miss > 0] if miss[miss > 0].any() else "No missing values")

print("\n=== Sampling weight ===")
print("Missing weights:", df["wt_sample_weight"].isnull().sum())

print("\n=== Outcome summary (national sample only) ===")
national = df[df["flag_dzaleka"] == 0]
print(f"Records in national sample : {len(national)}")
print(f"Stunting prevalence        : {national['outcome_stunted'].mean():.3f}")
print(f"Wasting prevalence         : {national['outcome_wasting'].mean():.3f}")
print(f"Anemia prevalence          : {national['outcome_anemia'].mean():.3f}")

print("\n=== All columns ===")
for col in df.columns:
    print(" ", col)

for col in df.select_dtypes(["category"]).columns:
    df[col] = df[col].astype(str)

df.to_parquet(DATA_PROCESSED / "model_dataset_v3.parquet", index=False)
print("\nSaved: model_dataset_v3.parquet")
print("Final shape:", df.shape)

=== Row count ===
5414

=== Missingness ===
whz_score           0.008
waz_score           0.001
hemoglobin_gdl      0.250
outcome_anemia      0.250
str_wealth_index    0.054
dtype: float64

=== Sampling weight ===
Missing weights: 0

=== Outcome summary (national sample only) ===
Records in national sample : 5122
Stunting prevalence        : 0.358
Wasting prevalence         : 0.021
Anemia prevalence          : 0.489

=== All columns ===
  id_cluster
  id_household
  id_child_index
  wt_sample_weight
  flag_dzaleka
  haz_score
  whz_score
  waz_score
  hemoglobin_gdl
  outcome_stunted
  outcome_wasting
  outcome_anemia
  imm_child_age_months
  imm_child_sex
  imm_birth_order
  imm_birth_interval
  imm_first_born
  imm_size_at_birth
  imm_had_diarrhea
  und_maternal_age
  und_maternal_edu_level
  und_maternal_edu_years
  und_maternal_weight_kg
  und_maternal_height_cm
  und_total_children
  und_household_size
  str_wealth_index
  str_residence
  str_region
  str_district
  str_religion

